# Pipeline + Model Persistence

A real model is never *just* the estimator. Before the classifier ever sees a row, that row is imputed, scaled, and encoded. If any of those preprocessing steps is fit on the wrong data (or gets forgotten at deploy time), your metrics lie and your production predictions drift.

This notebook builds one **`sklearn.Pipeline`** that bundles *all* preprocessing with the model into a single object, then shows how to **persist** it with `joblib` so the exact same transform-then-predict logic survives a save/load round trip.

We will:
1. Generate a **synthetic** table with a mix of numeric **and** categorical columns (plus a few missing values), then split it.
2. Build a `ColumnTransformer` (impute + scale numerics, impute + one-hot encode categoricals) and wrap it with a classifier in **one** `Pipeline`.
3. Fit on train, evaluate accuracy on test.
4. `joblib.dump` the whole fitted pipeline, `joblib.load` it back, and **assert** the reloaded object predicts *identically* — proving the round trip is lossless.

Everything runs offline on `numpy`, `pandas`, `scikit-learn`, `joblib`, and `matplotlib`, and is fully seeded.

In [ ]:
import os                                              # build the save path next to this notebook
import numpy as np                                     # random data + array equality checks
import pandas as pd                                    # the synthetic table lives in a DataFrame
import joblib                                          # save/load the fitted pipeline to/from disk
import matplotlib.pyplot as plt                        # one small bar chart at the end

from sklearn.model_selection import train_test_split   # split rows into train/test
from sklearn.pipeline import Pipeline                  # chains steps into ONE fit/predict object
from sklearn.compose import ColumnTransformer          # route different columns to different preprocessors
from sklearn.impute import SimpleImputer               # fill in the missing values
from sklearn.preprocessing import StandardScaler, OneHotEncoder  # scale numerics / encode categoricals
from sklearn.ensemble import RandomForestClassifier    # the classifier at the end of the pipeline
from sklearn.metrics import accuracy_score             # test-set scoring

# One global seed drives every random draw below (data generation, split, and the model),
# so re-running the notebook top-to-bottom always produces byte-identical results.
SEED = 42
rng = np.random.default_rng(SEED)   # modern NumPy Generator; we pass SEED to sklearn objects too

print("imports ok")

## 1. A synthetic dataset with mixed types

To make the preprocessing *non-trivial* we hand-build a table with three flavours of column:

- **Numeric**: `age`, `income`, `tenure_years` — different scales, so scaling matters.
- **Categorical**: `city`, `plan` — strings that must be encoded into numbers before a model can use them.
- **A sprinkle of missing values** in one numeric and one categorical column, so the imputers have real work to do.

The binary target `churn` is generated from a hidden linear rule on the features plus noise, so the signal is learnable but not trivial. Doing this by hand (rather than `make_classification`) is exactly what lets us inject strings and `NaN`s.

In [ ]:
N = 1200   # number of rows; small enough to train in well under a second

# --- Numeric features, each on a deliberately different scale ---
age = rng.integers(18, 70, size=N).astype(float)          # 18..69, integer years -> float so NaN can live here
income = rng.normal(60_000, 18_000, size=N)               # ~normal, thousands -> dominates raw distance w/o scaling
tenure_years = rng.exponential(4.0, size=N)               # skewed, small magnitude

# --- Categorical features drawn with uneven class probabilities (more realistic) ---
city = rng.choice(["NYC", "SF", "CHI", "AUS"], size=N, p=[0.4, 0.25, 0.2, 0.15])
plan = rng.choice(["basic", "pro", "enterprise"], size=N, p=[0.5, 0.35, 0.15])

# --- Hidden ground-truth rule: a linear score -> probability -> 0/1 label ---
# (Built BEFORE we punch holes in the data, so the label reflects the true signal.)
plan_boost = np.select(                                   # enterprise customers churn less
    [plan == "basic", plan == "pro", plan == "enterprise"],
    [0.6, 0.0, -0.8],
)
score = (
    0.03 * (age - 40)                # older -> slightly more likely to churn
    - 0.00002 * (income - 60_000)    # higher income -> less likely
    - 0.15 * tenure_years            # longer tenure -> much less likely
    + plan_boost
    + rng.normal(0, 0.5, size=N)     # irreducible noise so the task isn't perfectly separable
)
prob = 1.0 / (1.0 + np.exp(-score)) # squash the score into (0,1) with a logistic
churn = (prob > 0.5).astype(int)    # final binary target

# Assemble everything into a single DataFrame (how real tabular data actually arrives).
df = pd.DataFrame({
    "age": age,
    "income": income,
    "tenure_years": tenure_years,
    "city": city,
    "plan": plan,
    "churn": churn,
})

# --- Inject a few MISSING values so the imputers are genuinely needed ---
# Pick a random 4% of rows for 'income' (numeric) and 3% for 'city' (categorical) and blank them out.
missing_income = rng.choice(N, size=int(0.04 * N), replace=False)
missing_city = rng.choice(N, size=int(0.03 * N), replace=False)
df.loc[missing_income, "income"] = np.nan
df.loc[missing_city, "city"] = np.nan

print(f"shape: {df.shape}")
print(f"missing per column:\n{df.isna().sum()}\n")
print(f"target balance (churn):\n{df['churn'].value_counts(normalize=True).round(3)}")
df.head()

## 2. Train/test split

Split **before** touching the data with any transformer. Every statistic a preprocessor learns — the imputer's fill value, the scaler's mean/std, the encoder's set of known categories — must come from the **training rows only**. The test set has to stand in for genuinely unseen data.

We separate the feature columns `X` from the target `y`, and keep `X` as a **DataFrame** on purpose: the `ColumnTransformer` will select branches *by column name*, which is far more robust than positional indices.

In [ ]:
X = df.drop(columns="churn")   # features: 3 numeric + 2 categorical columns, still a DataFrame
y = df["churn"]                # target: the 0/1 Series we want to predict

# 25% held out for testing. stratify=y preserves the churn ratio in both splits;
# random_state ties the split to our global SEED for reproducibility.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

print(f"X_train: {X_train.shape}   X_test: {X_test.shape}")
print(f"columns : {list(X_train.columns)}")

## 3. A `ColumnTransformer` for the two column groups

Numeric and categorical columns need completely different treatment, so we build a small sub-pipeline for each and let a `ColumnTransformer` route the right columns to the right branch:

**Numeric branch** (`age`, `income`, `tenure_years`):
1. `SimpleImputer(strategy="median")` — fill blanks with the column median (robust to the skew/outliers in `income`).
2. `StandardScaler` — recenter to mean 0, variance 1 so no single large-magnitude column (e.g. `income` in the tens of thousands) dominates.

**Categorical branch** (`city`, `plan`):
1. `SimpleImputer(strategy="most_frequent")` — fill blanks with the most common category.
2. `OneHotEncoder(handle_unknown="ignore")` — turn each category into its own 0/1 column. `handle_unknown="ignore"` is the key safety valve: if a category never seen during training shows up at predict time, it is encoded as all-zeros instead of raising an error.

Crucially, every one of these steps *learns its parameters during `fit` on the training data only* and then merely *applies* them at transform time — which is exactly what prevents leakage.

In [ ]:
# Column groups, selected BY NAME so the transformer is robust to column reordering.
numeric_features = ["age", "income", "tenure_years"]
categorical_features = ["city", "plan"]

# Numeric branch: impute missing with the median, THEN standardize.
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),   # median is robust to income's skew/outliers
    ("scaler", StandardScaler()),                    # mean 0 / std 1 so scales are comparable
])

# Categorical branch: impute missing with the mode, THEN one-hot encode.
categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),           # fill NaN city with the commonest city
    ("onehot", OneHotEncoder(handle_unknown="ignore")),            # unseen categories -> all-zero row, no crash
])

# The ColumnTransformer runs both branches in parallel and concatenates their outputs.
# remainder="drop" (the default) means any column not listed is discarded — here that's none.
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
])

preprocessor   # rendering it shows the two-branch structure

## 4. Why wrap it all in ONE `Pipeline`?

Now we stack the `preprocessor` and a `RandomForestClassifier` into a single `Pipeline`. Two reasons this matters more than it first appears:

**1. No data leakage.** When you call `pipe.fit(X_train, y_train)`, the pipeline calls `fit_transform` on the preprocessing steps using *only* the training data, then feeds the result to the classifier. At `pipe.predict(X_test)` time it calls plain `transform` — reusing the medians, means, stds, and category lists learned from **train**. The test set never influences those statistics. If you instead scaled the *whole* dataset before splitting, information from the test rows would bleed into the training statistics and inflate your score.

**2. One object to deploy.** The fitted pipeline *is* the model. Preprocessing and prediction travel together as a single artifact. You hand raw, unprocessed DataFrame rows to `.predict()` and it does everything internally — there is no separate "remember to scale and encode the input the same way" script to keep in sync between training and production. That eliminates a whole class of train/serve skew bugs.

$$\text{raw } X \;\xrightarrow{\text{impute}}\; \xrightarrow{\text{scale / one-hot}}\; \xrightarrow{\text{RandomForest}}\; \hat{y}$$

In [ ]:
# The full model: preprocessing feeds directly into the classifier, as ONE estimator.
pipe = Pipeline(steps=[
    ("preprocess", preprocessor),   # step 1: the ColumnTransformer from the previous cell
    ("clf", RandomForestClassifier( # step 2: the actual classifier
        n_estimators=200,           # 200 trees: plenty for this small table, still fast
        max_depth=8,                # cap depth to curb overfitting and keep it snappy
        random_state=SEED,          # tie the forest's randomness to our global seed
        n_jobs=1,                   # single-threaded -> deterministic + no oversubscription warnings
    )),
])

# A single fit call: the preprocessor is fit_transform'd on X_train, then the forest is
# trained on the transformed matrix. All learned state stays inside `pipe`.
pipe.fit(X_train, y_train)

print("pipeline fitted")
pipe

## 5. Evaluate on the test set

We pass the **raw** `X_test` DataFrame — blanks, strings and all — straight to `pipe.predict`. The pipeline imputes, scales, and encodes it *using the training-set statistics*, then runs the forest. This is the payoff of the one-object design: prediction on new data is a single call.

In [ ]:
# Predict on raw test rows; the pipeline handles every preprocessing step internally.
y_pred = pipe.predict(X_test)

test_acc = accuracy_score(y_test, y_pred)   # fraction of test rows classified correctly
print(f"Test accuracy: {test_acc:.4f}")

## 6. Persistence with `joblib`

A fitted model only earns its keep if you can save it and reuse it later without retraining. We serialize the **entire fitted pipeline** — preprocessing state and the trained forest together — to a single file.

**Why `joblib` and not plain `pickle`?** `joblib` *is* a pickle-based format under the hood, but it is specifically optimized for objects that carry large NumPy arrays (exactly what scikit-learn models are: forests of arrays, scaler means, encoder categories). It stores those arrays efficiently and is the method scikit-learn's own documentation recommends. For pure-Python objects with no big arrays, plain `pickle` is equivalent.

**Cautions** (serialization is not magic):
- **Version pinning.** A pickle/joblib file is *not* a stable, portable format. Loading a model saved under a different scikit-learn / NumPy version can break or, worse, silently misbehave. Record the library versions alongside the file and load under a matching environment.
- **Never unpickle untrusted files.** Loading a joblib/pickle file can execute arbitrary code embedded in it. Only load artifacts you produced or fully trust — treat an untrusted `.joblib` like an untrusted executable.
- For long-term or cross-language needs, consider exporting to a neutral format (e.g. ONNX) instead of relying on pickle.

In [ ]:
# Build the save path NEXT TO this notebook so it works regardless of Jupyter's CWD.
# In a .ipynb __file__ isn't defined, so fall back to the current working directory.
try:
    HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:
    HERE = os.getcwd()

MODEL_PATH = os.path.join(HERE, "model_pipeline.joblib")

# Serialize the WHOLE fitted pipeline (preprocessor state + trained forest) to one file.
joblib.dump(pipe, MODEL_PATH)

# Report where it landed and how big it is (arrays dominate the size).
size_kb = os.path.getsize(MODEL_PATH) / 1024
print(f"saved to : {MODEL_PATH}")
print(f"file size: {size_kb:.1f} KB")

## 7. Load it back and prove the round trip is lossless

We `joblib.load` the file into a **fresh** variable — `loaded_pipe` — as if this were a brand-new Python process that never saw the training code. Then we predict with it on the same `X_test` and **assert** the predictions are *exactly* equal to the original pipeline's. If serialization dropped or corrupted any learned state (a scaler mean, an encoder category, a tree split), the assertion would fail. Exact equality here is the guarantee that what you deploy is byte-for-byte the model you trained.

In [ ]:
# Load into a fresh object — simulates a separate serving process with no training context.
loaded_pipe = joblib.load(MODEL_PATH)

# Predict again with the reloaded pipeline on the SAME raw test rows.
y_pred_loaded = loaded_pipe.predict(X_test)

# Hard proof the round trip lost nothing: every label must match the original exactly.
assert np.array_equal(y_pred, y_pred_loaded), "Reloaded predictions differ from the original!"

# Also confirm the class-probabilities are identical to full float precision (stricter check).
assert np.allclose(pipe.predict_proba(X_test), loaded_pipe.predict_proba(X_test)), "Probabilities differ!"

reloaded_acc = accuracy_score(y_test, y_pred_loaded)
print(f"Original test accuracy : {test_acc:.4f}")
print(f"Reloaded test accuracy : {reloaded_acc:.4f}")
print("Round-trip check PASSED: reloaded predictions match the original exactly.")

## 8. A quick look at what the forest learned

As a sanity check (and a small payoff for building the pipeline), we pull feature importances out of the trained forest. Note the feature names come from *inside* the pipeline: the one-hot encoder expanded `city` and `plan` into several 0/1 columns, and `get_feature_names_out()` recovers those names so the plot is readable.

In [ ]:
# Recover the post-transform feature names (numeric passthrough + expanded one-hot columns).
feat_names = loaded_pipe.named_steps["preprocess"].get_feature_names_out()
importances = loaded_pipe.named_steps["clf"].feature_importances_

# Sort by importance for a clean horizontal bar chart.
order = np.argsort(importances)             # ascending -> most important ends up at the top

plt.figure(figsize=(7, 4))
plt.barh(np.array(feat_names)[order], importances[order], color="#6699cc")
plt.title("Random forest feature importances (via the fitted pipeline)")
plt.xlabel("importance")
plt.tight_layout()
plt.show()

## Summary

- A **`Pipeline`** fuses preprocessing and the model into one estimator: `fit` learns all statistics from the training data only (no leakage), and `predict` re-applies them to raw new data automatically.
- A **`ColumnTransformer`** lets numeric and categorical columns follow different preprocessing paths, then concatenates the results. `OneHotEncoder(handle_unknown="ignore")` keeps prediction robust to unseen categories.
- **`joblib.dump` / `joblib.load`** persist the entire fitted pipeline as one artifact; the reloaded object produced **identical** predictions and probabilities — a lossless round trip.
- Persistence caveats: **pin library versions**, and **never unpickle untrusted files** — loading one can run arbitrary code.